This agent generates a search strings to interrogate Scopus database and retain all the litterature linked to the string.
The string is created collecting all the names linked to a scientific name according to EPPO Global Database.
The agent can connect to EPPO Api to get the unique code associated with a scientific name, collect all the names linked and can assemble
the research string.

REQUEST LIBRARIES

In [2]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [55]:
import requests
from langchain_core.tools import InjectedToolArg, tool
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
import os
from dotenv import load_dotenv, dotenv_values
import pandas as pd

 TOOLS

In [56]:
scientific_name = 'Coccus viridis'

In [93]:
#load .env variables
load_dotenv()

True

In [ ]:
@tool
def get_eppo_names(scientific_name:str):
    """"
    MANDATORY FIRST STEP. 
    Retrieves the full list of synonyms and common names from the EPPO database.
    You MUST call this tool even if you think you already know the names.
    Retrieves all common names for a given organism from the EPPO database.
    Use this tool whenever you need to know the names associated with a species.
    Input: A scientific name (e.g., 'Coccus viridis').
    Output: A list of strings containing all associated names.
    """
    eppo_token = os.getenv('EPPO_API_KEY')
    api_url = "https://api.eppo.int/gd/v2/tools/name2codes?name="
  
    #get the code from the api
    headers = {'Accept' : 'application/json', 'X-Api-Key' : eppo_token}
    response = requests.get(f'{api_url}{scientific_name}&onlyPreferred=true',headers=headers)
    response = response.json()[0]
    eppo_code = response['eppocode']

    names_url = "https://api.eppo.int/gd/v2/"
  
    #get the code from the api
    headers = {'Accept' : 'application/json', 'X-Api-Key' : eppo_token}
    response = requests.get(f'{names_url}taxons/taxon/{eppo_code}/names',headers=headers)
    response = response.json()
    names_data_frame = pd.DataFrame(response)
    list_of_names = list(names_data_frame['fullname'])
    return(list_of_names)
    

In [82]:
@tool
def create_search_string(list_of_names:list):
    """
    MANDATORY SECOND STEP.
    Converts the output of 'get_eppo_names' into a Scopus string.
    DO NOT call this tool using only the initial user input. 
    It requires a LIST of multiple names.
    Converts a list of names into a formatted Scopus search string.
    CRITICAL: This tool should ONLY be used after obtaining a list of names, 
    and ONLY if the user specifically requested a 'search string' or 'query'.
    Input: A list of strings.
    """
    raw_string = " OR ".join([f'"{name}"' for name in list_of_names])

    scopus_string = f"TITLE-ABS-KEY({raw_string})"

    return(scopus_string)

In [102]:
from langchain_groq import ChatGroq
groq_key=os.getenv('GROQ_API_KEY')
llm = ChatGroq(
    model="qwen/qwen3-32b", # Oppure "llama-3.1-70b-versatile"
    temperature=0,
    groq_api_key=groq_key
)

In [105]:
# 2. Inizializzazione
#llm = ChatOllama(model="qwen2.5:14b", temperature=0, num_thread=8) # Prova con metà dei tuoi core

# 3. Creazione Agente
agent = create_agent(llm, tools=[get_eppo_names,create_search_string],system_prompt="""You are a research assistant.
- If the user asks for 'names' of an organism, use 'get_eppo_names' and stop.
- If the user asks for a 'search string' or 'query', you MUST:
    1. Call 'get_eppo_names' first to get the list.
    2. Then call 'create_search_string' using that list.
- Answer in the same language as the user.""")

# 4. Esecuzione
input_data = {"messages": [("user", "create a search string with all the common names for Erwinia amylovora?")]}

print("--- Inizio Conversazione (Modalità Streaming) ---")

# Usiamo agent.stream per ricevere i pezzi (chunks) della conversazione in tempo reale
for chunk in agent.stream(input_data, stream_mode="values"):
    # Prende l'ultimo messaggio generato nel chunk attuale
    last_message = chunk["messages"][-1]
    
    # Lo stampa in modo leggibile non appena viene generato
    last_message.pretty_print()
    
    print("-" * 30) # Separatore visivo per capire i passaggi

--- Inizio Conversazione (Modalità Streaming) ---
================================ Human Message =================================

create a search string with all the common names for Erwinia amylovora?
------------------------------
================================== Ai Message ==================================
Tool Calls:
  get_eppo_names (fcfe0b343)
 Call ID: fcfe0b343
  Args:
    scientific_name: Erwinia amylovora
------------------------------
================================= Tool Message =================================
Name: get_eppo_names

["Erwinia amylovora", "اللفحة النارية", "ildsot", "Feuerbrand: Obstgehölze", "Apfelbrand", "Birnenbrand", "Bakterienfeuerbrand", "fireblight", "fire blight", "twig blight of apple", "fuego bacteriano", "tizón de fuego del manzano", "tizón de fuego del peral", "marchitez del peral", "marchitez del manzano", "niebla del peral", "niebla del manzano", "feu bactérien", "brûlure bactérienne du poirier", "brûlure bactérienne du pommier", "kerakh